# Sentinel-5P CH4 Data Fetcher - Sweden

This notebook fetches methane (CH4) data from Sentinel-5P satellite over Sweden (or any configured region).

## Quick Start
1. **Configure** your parameters in the cell below (dates, region, etc.)
2. **Authenticate** with Google Earth Engine
3. **Load** the satellite data collection
4. **Fetch** the data for your region
5. **Save** to CSV/JSON files

The data will be saved in the `../data/` directory.

In [1]:
# ============================================================================
# CONFIGURATION - Change these variables as needed
# ============================================================================

# Load environment variables from .env file
from dotenv import load_dotenv
import os

# Load .env file from parent directory
load_dotenv(dotenv_path='../.env')

# Earth Engine Project ID (update with your project)
EE_PROJECT = "quick-composite-408320"

# Region of interest
COUNTRY_NAME = 'Sweden'  # Change to any country name (e.g., 'Italy', 'France', 'Germany')

# Date range (YYYY-MM-DD format)
# Note: Winter months (Nov-Feb) often have no data over Sweden due to lack of sunlight
# Summer months (May-Aug) have best coverage
START_DATE = '2024-03-01'
END_DATE = '2024-03-07'

# Band to analyze
# ============================================================================
# Available Sentinel-5P bands for wildfire/air quality indicators:
# 
# Methane (CH4):
BAND_NAME = 'CH4_column_volume_mixing_ratio_dry_air'
DATASET_ID = 'COPERNICUS/S5P/OFFL/L3_CH4'
#
# Carbon Monoxide (CO):
# BAND_NAME = 'CO_column_number_density'
# DATASET_ID = 'COPERNICUS/S5P/OFFL/L3_CO'
#
# Nitrogen Dioxide (NO2):
# BAND_NAME = 'NO2_column_number_density'
# DATASET_ID = 'COPERNICUS/S5P/OFFL/L3_NO2'
#
# Formaldehyde (HCHO):
# BAND_NAME = 'tropospheric_HCHO_column_number_density'
# DATASET_ID = 'COPERNICUS/S5P/OFFL/L3_HCHO'
#
# Aerosol Absorbing Index (AAI) - for smoke/dust:
# BAND_NAME = 'absorbing_aerosol_index'
# DATASET_ID = 'COPERNICUS/S5P/OFFL/L3_AER_AI'
#
# Sulfur Dioxide (SO2):
# BAND_NAME = 'SO2_column_number_density'
# DATASET_ID = 'COPERNICUS/S5P/OFFL/L3_SO2'
# ============================================================================

# Spatial sampling configuration
# Options: 'region_mean' or 'sample_points'
SAMPLING_MODE = 'region_mean'  # 'region_mean' (faster), 'sample_points' (detailed/slower)

# If using 'sample_points', how many points to sample across the region
NUM_SAMPLE_POINTS = 50  # Will create a grid of ~50 points across the region

# AWS S3 Configuration - reads from .env file
ENABLE_S3_UPLOAD = True  # Set to True to enable S3 upload
AWS_BUCKET_NAME = 'caff-dump'
AWS_REGION = 'eu-central-1'
AWS_ACCESS_KEY_ID = os.getenv('AWS_ACCESS_KEY_ID')  # Read from .env
AWS_SECRET_ACCESS_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')  # Read from .env
S3_PREFIX = 'google-data/'  # Path prefix in S3 bucket

print(f"Configuration:")
print(f"  Region: {COUNTRY_NAME}")
print(f"  Date range: {START_DATE} to {END_DATE}")
print(f"  Band: {BAND_NAME}")
print(f"  Dataset: {DATASET_ID}")
print(f"  Sampling mode: {SAMPLING_MODE}")
if SAMPLING_MODE == 'sample_points':
    print(f"  Sample points: {NUM_SAMPLE_POINTS}")
if ENABLE_S3_UPLOAD:
    print(f"\n☁️  S3 Upload: ENABLED")
    print(f"  Bucket: {AWS_BUCKET_NAME}")
    print(f"  Region: {AWS_REGION}")
    print(f"  Credentials: {'✓ Loaded from .env' if AWS_ACCESS_KEY_ID else '✗ Missing - check .env file'}")
print(f"\n💡 Tip: Sentinel-5P needs sunlight, so summer dates work best for northern countries!")

Configuration:
  Region: Sweden
  Date range: 2024-03-01 to 2024-03-07
  Band: CH4_column_volume_mixing_ratio_dry_air
  Dataset: COPERNICUS/S5P/OFFL/L3_CH4
  Sampling mode: region_mean

☁️  S3 Upload: ENABLED
  Bucket: caff-dump
  Region: eu-central-1
  Credentials: ✗ Missing - check .env file

💡 Tip: Sentinel-5P needs sunlight, so summer dates work best for northern countries!


## 1. Configuration

**Change these variables to customize your data fetch:**

In [2]:
import ee

# Run `earthengine authenticate` in terminal first if not already authenticated

try:
    ee.Initialize(project=EE_PROJECT)
    print(f"✓ Earth Engine initialized with project: {EE_PROJECT}")
except Exception as e:
    print("Authentication needed. Running ee.Authenticate()...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print(f"✓ Earth Engine initialized with project: {EE_PROJECT}")

✓ Earth Engine initialized with project: quick-composite-408320


## 2. Authenticate with Earth Engine

Run this cell to authenticate with Google Earth Engine. A browser window may open for login.

In [3]:
# Load Sentinel-5P CH4 collection with date filter
collection = ee.ImageCollection(DATASET_ID) \
    .select(BAND_NAME) \
    .filterDate(START_DATE, END_DATE)

print(f"Loaded collection: {collection.size().getInfo()} images total")


Loaded collection: 85 images total


## 3. Load Satellite Data Collection

Loading the Sentinel-5P CH4 image collection for the specified date range...

In [4]:
# Load region geometry from FAO country boundaries
region_fc = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(
    ee.Filter.eq('ADM0_NAME', COUNTRY_NAME)
)
region_geom = region_fc.geometry()

print(f"Region loaded: {COUNTRY_NAME}")

# Create sample points if in sample mode
if SAMPLING_MODE == 'sample_points':
    # Create a grid of sample points across the region
    sample_points = ee.FeatureCollection.randomPoints(
        region=region_geom,
        points=NUM_SAMPLE_POINTS,
        seed=42  # For reproducibility
    )
    print(f"Created {NUM_SAMPLE_POINTS} sample points across {COUNTRY_NAME}")

# Functions ---------------------------------------------------------------
def filter_valid_images(ic, region, band):
    """Filter collection to only images with data over the region."""
    def add_pixel_count(img):
        count = img.select(band).reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=region,
            scale=10000,
            bestEffort=True,
            maxPixels=1e9
        ).get(band)
        return img.set('pixel_count', count)
    ic_with_counts = ic.map(add_pixel_count)
    return ic_with_counts.filter(ee.Filter.gt('pixel_count', 0))

def process_batch_region_mean(start_date, end_date, region, band):
    """Process batch - extract region-wide mean."""
    batch_collection = ee.ImageCollection(DATASET_ID) \
        .select(band) \
        .filterDate(start_date, end_date)
    
    valid_batch = filter_valid_images(batch_collection, region, band)
    
    def add_region_mean(img):
        mean_val = img.select(band).reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=10000,
            bestEffort=True,
            maxPixels=1e9
        ).get(band)
        return img.set({
            'date_str': img.date().format('YYYY-MM-dd'),
            'region_mean': mean_val
        })
    
    processed = valid_batch.map(add_region_mean)
    
    dates = processed.aggregate_array('date_str').getInfo()
    means = processed.aggregate_array('region_mean').getInfo()
    
    batch_results = []
    for date, mean in zip(dates, means):
        if mean is not None:
            batch_results.append({'date': date, 'mean_ch4': mean})
    
    return batch_results

def process_batch_sample_points(start_date, end_date, points, band):
    """Process batch - extract values at sample points with lat/lon."""
    batch_collection = ee.ImageCollection(DATASET_ID) \
        .select(band) \
        .filterDate(start_date, end_date)
    
    # Get list of images
    image_list = batch_collection.toList(batch_collection.size())
    num_images = batch_collection.size().getInfo()
    
    batch_results = []
    
    # Process each image separately
    for i in range(num_images):
        img = ee.Image(image_list.get(i))
        
        # Get the date for this image
        date_str = img.date().format('YYYY-MM-dd').getInfo()
        
        # Sample the image at all points
        sampled = img.sampleRegions(
            collection=points,
            scale=10000,
            geometries=True
        )
        
        # Get the results for this image
        sample_list = sampled.getInfo()
        
        # Extract data from each sample point
        for feature in sample_list['features']:
            props = feature['properties']
            coords = feature['geometry']['coordinates']
            
            ch4_value = props.get(band)
            if ch4_value is not None:
                batch_results.append({
                    'date': date_str,
                    'longitude': coords[0],
                    'latitude': coords[1],
                    'ch4': ch4_value
                })
    
    return batch_results

# Process data in monthly batches to avoid timeouts
print(f"\nProcessing data in monthly batches to avoid timeouts...")
print(f"Mode: {SAMPLING_MODE}")
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd

start = datetime.strptime(START_DATE, '%Y-%m-%d')
end = datetime.strptime(END_DATE, '%Y-%m-%d')

results = []
current = start
batch_num = 1

while current < end:
    # Calculate next month
    next_month = current + relativedelta(months=1)
    if next_month > end:
        next_month = end
    
    batch_start = current.strftime('%Y-%m-%d')
    batch_end = next_month.strftime('%Y-%m-%d')
    
    print(f"\nBatch {batch_num}: {batch_start} to {batch_end}")
    
    try:
        if SAMPLING_MODE == 'sample_points':
            batch_results = process_batch_sample_points(batch_start, batch_end, sample_points, BAND_NAME)
        else:
            batch_results = process_batch_region_mean(batch_start, batch_end, region_geom, BAND_NAME)
        
        results.extend(batch_results)
        print(f"  ✓ Found {len(batch_results)} observations")
    except Exception as e:
        print(f"  ⚠ Error: {e}")
    
    current = next_month
    batch_num += 1

print(f"\n{'='*60}")
print(f"✓ Complete! Found {len(results)} total observations")
if results:
    if SAMPLING_MODE == 'sample_points':
        print(f"  Date range: {min(r['date'] for r in results)} to {max(r['date'] for r in results)}")
        print(f"  Spatial points: {len(set((r['latitude'], r['longitude']) for r in results))}")
    else:
        print(f"  Date range: {results[0]['date']} to {results[-1]['date']}")
else:
    print(f"  No data found for {COUNTRY_NAME} in this period.")

Region loaded: Sweden

Processing data in monthly batches to avoid timeouts...
Mode: region_mean

Batch 1: 2024-03-01 to 2024-03-07

Batch 1: 2024-03-01 to 2024-03-07
  ✓ Found 6 observations

✓ Complete! Found 6 total observations
  Date range: 2024-03-03 to 2024-03-06
  ✓ Found 6 observations

✓ Complete! Found 6 total observations
  Date range: 2024-03-03 to 2024-03-06


In [5]:
# Preview the data
import pandas as pd

if results:
    df_preview = pd.DataFrame(results)
    print("Data Preview:")
    print(f"Shape: {df_preview.shape}")
    print(f"\nColumns: {list(df_preview.columns)}")
    print(f"\nFirst 10 rows:")
    print(df_preview.head(10))
    print(f"\nData types:")
    print(df_preview.dtypes)
    
    if SAMPLING_MODE == 'sample_points':
        print(f"\nSpatial coverage:")
        print(f"  Latitude range: {df_preview['latitude'].min():.2f} to {df_preview['latitude'].max():.2f}")
        print(f"  Longitude range: {df_preview['longitude'].min():.2f} to {df_preview['longitude'].max():.2f}")
        print(f"  Unique locations: {df_preview[['latitude', 'longitude']].drop_duplicates().shape[0]}")
else:
    print("No data to preview. Run the previous cell first.")

Data Preview:
Shape: (6, 2)

Columns: ['date', 'mean_ch4']

First 10 rows:
         date     mean_ch4
0  2024-03-03  1875.452150
1  2024-03-04  1874.723396
2  2024-03-04  1873.643209
3  2024-03-05  1886.531801
4  2024-03-06  1867.934157
5  2024-03-06  1839.941915

Data types:
date         object
mean_ch4    float64
dtype: object


## 5. Save Data Locally

Saves the fetched data to both **CSV** and **JSON** formats in the `../data/` directory.

- **CSV**: Simple table (date, mean_ch4)
- **JSON**: Includes metadata + data

In [6]:
# Save the raw results to CSV and JSON
import json
import os
from datetime import datetime

if results:
    # Create data directory if it doesn't exist
    data_dir = '../data'
    os.makedirs(data_dir, exist_ok=True)
    
    # Generate filename with timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    mode_suffix = '_points' if SAMPLING_MODE == 'sample_points' else '_mean'
    filename_base = f"ch4_{COUNTRY_NAME.lower().replace(' ', '_')}_{START_DATE}_to_{END_DATE}{mode_suffix}"
    
    # Save as CSV
    csv_path = os.path.join(data_dir, f"{filename_base}.csv")
    import pandas as pd
    results_df = pd.DataFrame(results)
    results_df.to_csv(csv_path, index=False)
    print(f"✓ Saved to CSV: {csv_path}")
    
    # Save as JSON
    json_path = os.path.join(data_dir, f"{filename_base}.json")
    
    metadata = {
        'country': COUNTRY_NAME,
        'start_date': START_DATE,
        'end_date': END_DATE,
        'band': BAND_NAME,
        'dataset': DATASET_ID,
        'sampling_mode': SAMPLING_MODE,
        'observations': len(results),
        'fetched_at': timestamp
    }
    
    if SAMPLING_MODE == 'sample_points':
        metadata['num_sample_points'] = NUM_SAMPLE_POINTS
        metadata['unique_locations'] = len(set((r['latitude'], r['longitude']) for r in results))
    
    with open(json_path, 'w') as f:
        json.dump({
            'metadata': metadata,
            'data': results
        }, f, indent=2)
    print(f"✓ Saved to JSON: {json_path}")
    
    print(f"\n📊 Summary:")
    print(f"   Total observations: {len(results)}")
    if results:
        if SAMPLING_MODE == 'sample_points':
            print(f"   Columns: date, latitude, longitude, ch4")
            print(f"   Unique locations: {len(set((r['latitude'], r['longitude']) for r in results))}")
        else:
            print(f"   Columns: date, mean_ch4")
        print(f"   Date range: {min(r['date'] for r in results)} to {max(r['date'] for r in results)}")
    print(f"   Files saved in: {os.path.abspath(data_dir)}")
else:
    print("⚠ No data to save. Run the previous cell first or adjust the date range.")

✓ Saved to CSV: ../data/ch4_sweden_2024-03-01_to_2024-03-07_mean.csv
✓ Saved to JSON: ../data/ch4_sweden_2024-03-01_to_2024-03-07_mean.json

📊 Summary:
   Total observations: 6
   Columns: date, mean_ch4
   Date range: 2024-03-03 to 2024-03-06
   Files saved in: /home/kemal/Desktop/uni-material/Caffein/data


## 6. Upload to AWS S3 (Optional)

Upload the saved CSV and JSON files to your AWS S3 bucket.

In [7]:
# Upload to AWS S3
if ENABLE_S3_UPLOAD and results:
    try:
        import boto3
        from botocore.exceptions import ClientError
        import os
        
        print("Uploading to AWS S3...")
        
        # Initialize S3 client
        # Option 1: Use credentials from config (if provided)
        if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
            s3_client = boto3.client(
                's3',
                region_name=AWS_REGION,
                aws_access_key_id=AWS_ACCESS_KEY_ID,
                aws_secret_access_key=AWS_SECRET_ACCESS_KEY
            )
        # Option 2: Use environment variables or AWS credentials file
        else:
            s3_client = boto3.client('s3', region_name=AWS_REGION)
        
        # Upload CSV file
        csv_filename = os.path.basename(csv_path)
        csv_s3_key = f"{S3_PREFIX}{csv_filename}"
        
        with open(csv_path, 'rb') as f:
            s3_client.put_object(
                Bucket=AWS_BUCKET_NAME,
                Key=csv_s3_key,
                Body=f.read(),
                ContentType='text/csv'
            )
        print(f"✓ Uploaded CSV to s3://{AWS_BUCKET_NAME}/{csv_s3_key}")
        
        # Upload JSON file
        json_filename = os.path.basename(json_path)
        json_s3_key = f"{S3_PREFIX}{json_filename}"
        
        with open(json_path, 'rb') as f:
            s3_client.put_object(
                Bucket=AWS_BUCKET_NAME,
                Key=json_s3_key,
                Body=f.read(),
                ContentType='application/json'
            )
        print(f"✓ Uploaded JSON to s3://{AWS_BUCKET_NAME}/{json_s3_key}")
        
        print(f"\n☁️  S3 Upload Complete!")
        print(f"   Bucket: {AWS_BUCKET_NAME}")
        print(f"   Region: {AWS_REGION}")
        print(f"   Files:")
        print(f"     - {csv_s3_key}")
        print(f"     - {json_s3_key}")
        
    except ImportError:
        print("⚠ boto3 not installed. Install with: pip install boto3")
    except ClientError as e:
        print(f"⚠ AWS Error: {e}")
        print("\nTroubleshooting:")
        print("  1. Check your AWS credentials are correct")
        print("  2. Verify bucket name and region")
        print("  3. Ensure your IAM user has PutObject permissions")
    except Exception as e:
        print(f"⚠ Upload failed: {e}")
        
elif ENABLE_S3_UPLOAD and not results:
    print("⚠ No data to upload. Run the data fetching cell first.")
elif not ENABLE_S3_UPLOAD:
    print("ℹ️  S3 upload is disabled. Set ENABLE_S3_UPLOAD = True in the configuration cell to enable.")

Uploading to AWS S3...
✓ Uploaded CSV to s3://caff-dump/google-data/ch4_sweden_2024-03-01_to_2024-03-07_mean.csv
✓ Uploaded CSV to s3://caff-dump/google-data/ch4_sweden_2024-03-01_to_2024-03-07_mean.csv
✓ Uploaded JSON to s3://caff-dump/google-data/ch4_sweden_2024-03-01_to_2024-03-07_mean.json

☁️  S3 Upload Complete!
   Bucket: caff-dump
   Region: eu-central-1
   Files:
     - google-data/ch4_sweden_2024-03-01_to_2024-03-07_mean.csv
     - google-data/ch4_sweden_2024-03-01_to_2024-03-07_mean.json
✓ Uploaded JSON to s3://caff-dump/google-data/ch4_sweden_2024-03-01_to_2024-03-07_mean.json

☁️  S3 Upload Complete!
   Bucket: caff-dump
   Region: eu-central-1
   Files:
     - google-data/ch4_sweden_2024-03-01_to_2024-03-07_mean.csv
     - google-data/ch4_sweden_2024-03-01_to_2024-03-07_mean.json
